# 6.2 — Cost Center Variance Dashboard

## 1. Overview

This notebook produces a Tableau extract for cost-center variance reporting — the kind of view a controller would build from SAP transaction **KSB1** (line-item actuals) plus the corresponding plan data.

**Input:** `data/raw/cost_center_actuals.csv` (registered as `cost_centers`).

**Output:** `data/tableau/cost_center_variance.hyper`.

## 2. Load & inspect

In [ ]:
from analytics_bootcamp.duckdb_utils import get_connection, query_df
import pandas as pd

con = get_connection()
df = query_df("SELECT * FROM cost_centers", con)
print(df.shape)
df.head()

In [ ]:
df.dtypes

## 3. Clean

In [ ]:
df = df.copy()
for col in ["ACTUAL_AMT", "PLAN_AMT"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
df["GJAHR"] = df["GJAHR"].astype(int)
df["PERIOD"] = df["PERIOD"].astype(int)
df.isna().sum()

## 4. Analytical layer

In [ ]:
variance_by_cc = con.sql("""
    SELECT
        KOSTL,
        SUM(ACTUAL_AMT)            AS actual,
        SUM(PLAN_AMT)              AS plan,
        SUM(ACTUAL_AMT - PLAN_AMT) AS variance,
        ROUND(
            SUM(ACTUAL_AMT - PLAN_AMT) / NULLIF(SUM(PLAN_AMT), 0) * 100,
            2
        ) AS variance_pct
    FROM cost_centers
    GROUP BY KOSTL
    ORDER BY variance DESC
""").df()
variance_by_cc.head(10)

In [ ]:
over_plan = con.sql("""
    SELECT
        KOSTL,
        SUM(ACTUAL_AMT - PLAN_AMT) AS variance
    FROM cost_centers
    GROUP BY KOSTL
    HAVING SUM(ACTUAL_AMT) > SUM(PLAN_AMT)
    ORDER BY variance DESC
""").df()
over_plan

In [ ]:
period_trend = con.sql("""
    SELECT
        GJAHR,
        PERIOD,
        SUM(ACTUAL_AMT) AS actual,
        SUM(PLAN_AMT)   AS plan
    FROM cost_centers
    GROUP BY GJAHR, PERIOD
    ORDER BY GJAHR, PERIOD
""").df()
period_trend.head()

In [ ]:
by_element = con.sql("""
    SELECT
        KSTAR,
        SUM(ACTUAL_AMT) AS actual,
        SUM(PLAN_AMT)   AS plan,
        SUM(ACTUAL_AMT - PLAN_AMT) AS variance
    FROM cost_centers
    GROUP BY KSTAR
    ORDER BY variance DESC
""").df()
by_element.head(10)

## 5. Visualize

A waterfall-style horizontal bar of variance per cost center, plus the period trend of actual vs plan.

In [ ]:
import matplotlib.pyplot as plt

top = variance_by_cc.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#c0504d" if v > 0 else "#4f81bd" for v in top["variance"]]
ax.barh(top["KOSTL"].astype(str), top["variance"], color=colors)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_title("Variance by Cost Center (red = over plan)")
ax.set_xlabel("Variance (Actual - Plan)")
plt.tight_layout()
plt.show()

In [ ]:
period_trend["label"] = (
    period_trend["GJAHR"].astype(str) + "-" + period_trend["PERIOD"].astype(str).str.zfill(2)
)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(period_trend["label"], period_trend["actual"], marker="o", label="Actual")
ax.plot(period_trend["label"], period_trend["plan"],   marker="o", label="Plan")
ax.set_title("Actual vs Plan by Period")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Export to .hyper

In [ ]:
from analytics_bootcamp.tableau import to_hyper

extract = con.sql("""
    SELECT
        KOSTL,
        GJAHR,
        PERIOD,
        KSTAR,
        SUM(ACTUAL_AMT)            AS actual,
        SUM(PLAN_AMT)              AS plan,
        SUM(ACTUAL_AMT - PLAN_AMT) AS variance,
        ROUND(
            SUM(ACTUAL_AMT - PLAN_AMT) / NULLIF(SUM(PLAN_AMT), 0) * 100,
            2
        ) AS variance_pct
    FROM cost_centers
    GROUP BY KOSTL, GJAHR, PERIOD, KSTAR
""").df()

to_hyper(extract, "data/tableau/cost_center_variance.hyper", "Cost_Center_Variance")

## 7. Tableau tips

**Management reporting layout** — controllers tend to look at variance in three lenses:

1. **Where is the money going?** Bar chart of `variance` by `KOSTL` (cost center), color-encoded red/blue for over/under plan.
2. **Is it getting worse over time?** Dual-line chart of `actual` vs `plan` by `PERIOD` with `GJAHR` as a filter.
3. **What is driving it?** Treemap of `variance` by `KSTAR` (cost element) inside the selected cost center.

**Filters to add:** `GJAHR`, `KOSTL`, `KSTAR`. Wire dashboard actions so clicking a cost center filters both the period trend and the cost-element treemap.